In [ ]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document  # 确保导入了 Document

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

folder_path = "../data/"
all_docs = [] 

print(f"开始扫描 {folder_path} 目录下的txt文件")

# 1. 遍历文件夹，读取文件
for filename in os.listdir(folder_path):
    if filename.endswith(".txt"):
        file_path = os.path.join(folder_path, filename)
        print(f"   -> 发现文件: {filename}，正在读取...")
        
        # 准备不同的文件编码格式
        keys = ["utf-8", "utf-8-sig", "gb18030", "gbk", "gb2312"]
        success = False
        
        for key in keys:
            try:
                loader = TextLoader(file_path, encoding=key)
                docs = loader.load() # 得到 Document 对象列表
                all_docs.extend(docs) # 将当前文件的 Document 对象添加到总列表中
                
                print(f" 成功使用 [{key}] 编码读取")
                success = True
                break # 成功了就不试其他编码了，直接跳出循环
            except Exception:
                continue # 如果当前编码失败了，就继续试下一个编码
                
        # 如果所有编码都失败了，就启动暴力推土机模式
        if not success:
            print(f"  警告：{filename} 编码混乱，启动暴力推土机模式...")
            # 用原生 Python 强行读取，过滤死字节
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                raw_text = f.read()
            # 直接把整个文本当成一个 Document 对象，只能入库，无法进行切分
            doc = Document(page_content=raw_text, metadata={"source": file_path})
            all_docs.append(doc)

print(f" 文件扫描完毕！一共读取了 {len(all_docs)} 份文档。")

# 2. 切分和入库 
print(" 正在进行统一文本切分...")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = text_splitter.split_documents(all_docs)
print(f" 成功！所有文件被切分成了 {len(chunks)} 个知识块。")

print(" 正在加载本地 Embedding 模型...")
embeddings = HuggingFaceEmbeddings()

print(f" 正在将 {len(chunks)} 个知识块转化为向量并写入 Chroma 数据库...")
db = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory="../db")
print(" 所有文件批量入库彻底完成！")

📡 开启雷达，开始扫描 ../data/ 目录下的所有 txt 文件...
   -> 发现新文件: 个人信息保护法.txt，正在尝试读取...
      ✅ 成功使用 [utf-8] 编码读取！
   -> 发现新文件: 个人信息安全规范.txt，正在尝试读取...
      ✅ 成功使用 [gb18030] 编码读取！
   -> 发现新文件: 数据安全法.txt，正在尝试读取...
      ✅ 成功使用 [utf-8] 编码读取！
✅ 文件扫描完毕！一共读取了 3 份大文档。
✂️ 正在进行统一文本切分...
✅ 成功！所有文件被切分成了 131 个知识块。
🔌 正在加载本地 Embedding 模型...


C:\Users\smf\AppData\Local\Temp\ipykernel_11580\924945443.py:57: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
C:\Users\smf\AppData\Local\Temp\ipykernel_11580\924945443.py:57: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⏳ 正在将 131 个知识块转化为向量并写入 Chroma 数据库！(请耐心等待...)
🎉 所有文件批量入库彻底完成！


In [2]:
# 4. 模拟一次真实业务中的 RAG 检索
query = "什么是敏感个人信息？"
print(f"【测试问题】: {query}\n")

# 让 Chroma 数据库去寻找与 query 向量空间最接近的 2 个数据块 (k=5)
results = db.similarity_search(query, k=5) 

for i, res in enumerate(results):
    print(f"--- 💡 数据库检索到的第 {i+1} 条相关依据 ---")
    print(res.page_content)
    print("-" * 40)

【测试问题】: 什么是敏感个人信息？

--- 💡 数据库检索到的第 1 条相关依据 ---
当个人信息控制者停止运营其产品或服务时，应：
a) 及时停止继续收集个人信息；
b) 将停止运营的通知以逐一送达或公告的形式通知个人信息主体；
c) 对其所持有的个人信息进行删除或匿名化处理。
7 个人信息的使用
7.1 个人信息访问控制措施
对个人信息控制者的要求包括：
a) 对被授权访问个人信息的人员，应建立最小授权的访问控制策略，使其只能访问职责所需的最小必要的个人信息，且仅具备完成职责所需的最少的数据操作权限；
b) 对个人信息的重要操作设置内部审批流程，如进行批量修改、拷贝、下载等重要操作；
c) 对安全管理人员、数据操作人员、审计人员的角色进行分离设置；
----------------------------------------
--- 💡 数据库检索到的第 2 条相关依据 ---
注1：个人敏感信息包括身份证件号码、个人生物识别信息、银行账户、通信记录和内容、财产信息、征信信息、行踪轨迹、住宿信息、健康生理信息、交易信息、14岁以下（含）儿童的个人信息等。
注2：关于个人敏感信息的判定方法和类型参见附录B。
注3：个人信息控制者通过个人信息或其他信息加工处理后形成的信息，如一旦泄露、非法提供或滥用可能危害人身和财产安全，极易导致个人名誉、身心健康受到损害或歧视性待遇等的，属于个人敏感信息。
3.3 个人信息主体 personal information subject
个人信息所标识或者关联的自然人
----------------------------------------
--- 💡 数据库检索到的第 3 条相关依据 ---
对个人信息控制者的要求包括：
a) 应制定个人信息保护政策，内容应包括但不限于：
1) 个人信息控制者的基本情况，包括主体身份、联系方式。
2) 收集、使用个人信息的业务功能，以及各业务功能分别收集的个人信息类型。涉及个人敏感信息的，需明确标识或突出显示。
3) 个人信息收集方式、存储期限、涉及数据出境情况等个人信息处理规则。
4) 对外共享、转让、公开披露个人信息的目的、涉及的个人信息类型、接收个人信息的第三方类型，以及各自的安全和法律责任。
----------------------------------------

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
import os

load_dotenv("api_key.env", override=True) # 加载 .env 文件 同时重置环境变量

# 1. 唤醒大模型 
print("正在连接 DeepSeek 大脑...")
llm = ChatOpenAI(
    model="deepseek-chat", 
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    max_tokens=1024,
    temperature=0.1 # 设置为0.1让它回答更严谨，不要瞎发散
)

# 2. 把刚才检索到的 2 条法律依据拼成一段大白话
context_text = "\n".join([doc.page_content for doc in results])

# 3. 构建 CoT 提示词 展示模型思维链
prompt_template = f"""
你是一个顶级的数据安全合规专家。
请根据以下我为你检索到的《个人信息保护法》相关条文：
【参考依据开始】
{context_text}
【参考依据结束】

请对我提供的以下业务字段进行分类分级判定：
- 字段名称：user_health_record
- 业务描述：患者在医院的既往病史和生理健康检测记录

请严格按照以下步骤思考并输出结果：
Step 1: 语义拆解该字段代表的实体性质。
Step 2: 将其与参考依据中的定义进行比对。
Step 3: 给出最终的分类和分级结论（如 L1-L4）。
"""

# 4. 发送给大模型并打印结果
print("正在请大模型进行推理分析，请稍候...\n")
response = llm.invoke(prompt_template)

print("🌟 判定结果出炉：")
print(response.content)

正在连接 DeepSeek 大脑...
正在请大模型进行推理分析，请稍候...

🌟 判定结果出炉：
好的，作为顶级数据安全合规专家，我将严格遵循您提供的思考步骤，对字段 `user_health_record` 进行分析判定。

### **Step 1: 语义拆解该字段代表的实体性质**
- **字段名称**：`user_health_record`
- **业务描述**：患者在医院的既往病史和生理健康检测记录。
- **语义拆解**：
    1.  **主体**：该信息关联的主体是“患者”，即自然人，符合《个人信息保护法》中“个人信息主体”的定义。
    2.  **内容**：包含“既往病史”和“生理健康检测记录”。
        - **既往病史**：指个人过去所患疾病、诊疗过程、健康状况的历史信息。
        - **生理健康检测记录**：指通过医疗检查、检验等手段获得的，反映个人身体机能、生理指标和健康状况的数据。
    3.  **属性**：该信息直接、具体地描述了个人的身体健康状况，属于个人核心隐私范畴。

### **Step 2: 将其与参考依据中的定义进行比对**
根据您提供的参考依据，进行逐项比对：
1.  **是否属于“个人信息”**：该字段直接关联到特定的自然人（患者），能够单独或者与其他信息结合识别该自然人的身份，并描述其健康特征。完全符合“个人信息”的定义。
2.  **是否属于“个人敏感信息”**：参考依据中明确列举了“健康生理信息”属于个人敏感信息。同时，注3给出了判定原则：“一旦泄露、非法提供或滥用可能危害人身和财产安全，极易导致个人名誉、身心健康受到损害或歧视性待遇等”。
    - **比对分析**：
        - **直接对应**：字段内容“生理健康检测记录”与列举的“健康生理信息”完全吻合。
        - **风险符合**：“既往病史”和详细的健康记录一旦泄露，极可能导致个人遭受歧视（如就业、保险）、造成精神压力和心理伤害，甚至被用于精准诈骗等，严重危害个人权益。这完全符合注3所述的高风险情形。
    - **结论**：该字段信息属于法规定义中的 **“个人敏感信息”**。

### **Step 3: 给出最终的分类和分级结论**
- **分类结论**：**个人敏感信息**。
- **分级结

In [ ]:
import pandas as pd
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# 1. 加载环境变量并唤醒大模型
load_dotenv("api_key.env", override=True)
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"), 
    max_tokens=1024,
    temperature=0.1 
)

# 2. 连接 Chroma 法律知识库
print("正在连接本地法律知识库...")
local_model_path = r"G:\AI_Models\sentence-transformer"

embeddings = HuggingFaceEmbeddings(
    model_name=local_model_path,
    model_kwargs={'local_files_only': True} 
)
db = Chroma(persist_directory="../db", embedding_function=embeddings)

# 3. 读取准备的测试数据
print("正在读取 Excel 测试数据...")

excel_path = "../testdata/财务与会计.xlsx" 
！
# Pandas 读取 Excel 文件
df = pd.read_excel(excel_path)

# 进行测试
test_df = df

# 创建一个空列表，用来装大模型的判定结果
results_list = []

# 4. 开启自动化引擎：遍历每一行数据
print(f" 开始批量测试，共计 {len(test_df)} 条数据...\n")

for index, row in test_df.iterrows():
    field_en = row['英文名']
    field_cn = row['中文名']
    desc = row['描述']
    
    print(f"👉 正在处理: {field_cn} ({field_en})...")
    
    #  A：去知识库捞法律依据
    query = f"关于 {field_cn} {desc} 的数据安全和保密级别规定"
    docs = db.similarity_search(query, k=2)
    context_text = "\n".join([doc.page_content for doc in docs])
    
    #  B：组装 CoT 思维链 Prompt
    prompt_template = f"""
    你是一个顶级的数据安全合规专家。
    请根据以下我为你检索到的法律条文或行业规范：
    【参考依据开始】
    {context_text}
    【参考依据结束】

    请对我提供的以下业务字段进行分类分级判定：
    - 字段英文名：{field_en}
    - 字段中文名：{field_cn}
    - 业务描述：{desc}

    请严格按照以下步骤思考：
    Step 1 (语义分析): 该字段在真实业务中代表什么实体。
    Step 2 (合规比对): 将其与参考依据中的定级标准进行比对,并指出参考依据中的法律条文或规范名称。
    Step 3 (输出结论): 给出最终的分类和分级结论（如：L1一般数据，L2内部数据，L3敏感数据，L4核心数据）。
    """
    
    # 环节 C：连接DeepSeek 
    response = llm.invoke(prompt_template)
    
    # 将结果保存到我们的列表中
    results_list.append({
        "英文名": field_en,
        "中文名": field_cn,
        "业务描述": desc,
        "大模型分级结论": response.content
    })

# 5. 导出最终成果
print("\n 批量测试完成！正在生成结果报告...")
result_df = pd.DataFrame(results_list)

# 将结果保存为一个新的 CSV 文件
result_df.to_csv("../testdata/测试结果报告.csv", index=False, encoding='utf-8-sig')
print("报告已生成，请在左侧目录查看 测试结果报告.csv！")

正在连接本地法律知识库...


C:\Users\smf\AppData\Local\Temp\ipykernel_35792\2190558636.py:24: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: G:\AI_Models\sentence-transformer
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\smf\AppData\Local\Temp\ipykernel_35792\2190558636.py:28: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  db = Chroma(persist_directory="../db", embedding_function=embeddings)


正在读取 Excel 测试数据...
🚀 开始批量测试，共计 67 条数据...

👉 正在处理: 总账标识 (general_ledger_id)...
👉 正在处理: 科目代码 (account_code)...
👉 正在处理: 科目名称 (account_name)...
👉 正在处理: 科目类型 (account_type)...
👉 正在处理: 父级科目代码 (parent_account_code)...
👉 正在处理: 余额方向 (balance_direction)...
👉 正在处理: 期初余额 (period_beginning_balance)...
👉 正在处理: 期末余额 (period_ending_balance)...
👉 正在处理: 本期借方发生额 (current_debit_total)...
👉 正在处理: 本期贷方发生额 (current_credit_total)...
👉 正在处理: 是否现金科目 (is_cash_account)...
👉 正在处理: 是否备抵科目 (is_contra_account)...
👉 正在处理: 凭证号 (voucher_id)...
👉 正在处理: 凭证日期 (voucher_date)...
👉 正在处理: 凭证类型 (voucher_type)...
👉 正在处理: 分录行号 (entry_line_number)...
👉 正在处理: 借方金额 (debit_amount)...
👉 正在处理: 贷方金额 (credit_amount)...
👉 正在处理: 摘要 (description)...
👉 正在处理: 参考单据 (reference_document)...
👉 正在处理: 制单人 (prepared_by)...
👉 正在处理: 审核人 (reviewed_by)...
👉 正在处理: 过账状态 (posted_status)...
👉 正在处理: 过账日期 (posting_date)...
👉 正在处理: 应收应付类型 (ar_ap_type)...
👉 正在处理: 往来单位编号 (counterparty_id)...
👉 正在处理: 往来单位名称 (counterparty_name)...
👉 正在处理: 发票号码 (invoice_number)...
